In [1]:
import os
from dotenv import load_dotenv
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.messages import trim_messages



load_dotenv()

API_KEY = os.getenv("GOOGLE_API_KEY")

if not API_KEY:
    raise ValueError("GOOGLE_API_KEY not found.")



model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=API_KEY,
    temperature=0,
)



class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


trimmer = trim_messages(
    max_tokens=1000,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
)

def call_model(state: State):
    trimmed_messages = trimmer.invoke(state["messages"])

    if not trimmed_messages:
        trimmed_messages = state["messages"]

    response = model.invoke(trimmed_messages)
    return {"messages": [response]}



workflow = StateGraph(State)

workflow.add_node("chatbot", call_model)

workflow.add_edge(START, "chatbot")
workflow.add_edge("chatbot", END)

memory = MemorySaver()

app = workflow.compile(checkpointer=memory)



config = {
    "configurable": {
        "thread_id": "user_session_1"
    }
}

questions = [
    "Hi, my name is ALI HAIDER",
    "I am from Pakistan.",
    "I like Python programming.",
    "What is my name?",
    "Which country am I from?",
    "What do I like?"
]

for q in questions:
    result = app.invoke(
        {
            "messages": [
                HumanMessage(content=q)
            ]
        },
        config=config,
    )

    print(f"\nUser: {q}")
    print(f"AI: {result['messages'][-1].content}")


User: Hi, my name is ALI HAIDER
AI: Hi Ali Haider, nice to meet you! How can I help you today?

User: I am from Pakistan.
AI: That's great! Pakistan is a beautiful country with a rich history and culture.

How can I help you today, Ali Haider? Is there anything specific you'd like to discuss or know?

User: I like Python programming.
AI: That's excellent, Ali Haider! Python is a fantastic choice. It's incredibly versatile, easy to learn, and has a huge community.

What aspects of Python do you enjoy the most? Are you into web development, data science, automation, or something else?

User: What is my name?
AI: Your name is **Ali Haider**.

User: Which country am I from?
AI: You are from **Pakistan**.

User: What do I like?
AI: You like **Python programming**.
